In [6]:
"""
Exploratory analysis: production-unit openings/closings vs. municipal outcomes.

Produces four standalone HTML files in ./outputs/:
  1. national_timeseries.html   — openings, closings, net change over time
  2. active_stock.html          — active unit stock per year, nationally + by region
  3. choropleth_net_change.html — map of cumulative net change per kommune (normalized)
  4. top_movers.html            — ranking table of top municipalities by net change

All charts are interactive Plotly figures suitable for embedding on a static site.
"""
from __future__ import annotations

import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
UNITS_PATH = Path("../data/CVR_API/cvr_production_units_geocoded.csv")
STATS_PATH = Path("../data/geography/statsbank_combined.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

YEAR_START, YEAR_END = 2008, 2024
MIN_DURATION_DAYS = 30   # filter same-day-or-very-short units (registration artifacts)

# ----------------------------------------------------------------------------
# 1. Load and clean
# ----------------------------------------------------------------------------
def load_units(path: Path) -> pd.DataFrame:
    """Load production units, parse the two different date formats, clean."""
    df = pd.read_csv(
        path,
        dtype={"KommuneCode": str, "RegionCode": str, "UnitZipcode": str},
        low_memory=False,
    )
    # Drop the unnamed index column if present
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

    # Pad KommuneCode to 4 digits (e.g. "223" -> "0223") for joins with GeoJSON
    df["KommuneCode"] = df["KommuneCode"].str.zfill(4)

    # Parse dates. Start uses "15/06 - 1991", end uses ISO "2010-02-23".
    df["StartDate"] = pd.to_datetime(
        df["UnitStartdate"].str.replace(" ", "", regex=False),
        format="%d/%m-%Y", errors="coerce",
    )
    df["EndDate"] = pd.to_datetime(df["UnitEnddate"], errors="coerce")

    # Filter registration artifacts (open and close within MIN_DURATION_DAYS)
    duration = (df["EndDate"] - df["StartDate"]).dt.days
    artifact = duration.notna() & (duration < MIN_DURATION_DAYS)
    n_drop = int(artifact.sum())
    df = df.loc[~artifact].copy()
    print(f"  dropped {n_drop:,} short-lived registration artifacts (<{MIN_DURATION_DAYS} days)")

    # Filter out specific company names
    if 'CompanyName' in df.columns:
        n_before = len(df)
        # Remove municipalities and regions 
        df = df[~df['CompanyName'].str.contains('Kommune|Region', case=False, na=False)]
        # Remove supermarkets and retail chains
        df = df[~df['CompanyName'].str.contains('Salling|365discount|COOP|POST|politi|COMPASS', case=False, na=False)]
        print(f"  dropped {n_before - len(df):,} units based on company name filters")

    df["StartYear"] = df["StartDate"].dt.year
    df["EndYear"] = df["EndDate"].dt.year
    return df


def load_stats(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["KommuneCode"] = df["municipality_code"].astype(int).map(lambda x: f"{x:04d}")
    return df


print("Loading data...")
units = load_units(UNITS_PATH)
stats = load_stats(STATS_PATH)
print(f"  {len(units):,} units, {stats['KommuneCode'].nunique()} municipalities in stats")

# ----------------------------------------------------------------------------
# 2. Compute openings, closings, active stock per (kommune, year)
# ----------------------------------------------------------------------------
years = list(range(YEAR_START, YEAR_END + 1))

# Openings: count by start year
opens = (
    units[units["StartYear"].between(YEAR_START, YEAR_END)]
    .groupby(["KommuneCode", "StartYear"]).size()
    .rename("openings").reset_index().rename(columns={"StartYear": "year"})
)

# Closings: count by end year
closes = (
    units[units["EndYear"].between(YEAR_START, YEAR_END)]
    .groupby(["KommuneCode", "EndYear"]).size()
    .rename("closings").reset_index().rename(columns={"EndYear": "year"})
)

# Active stock at year-end t: started on/before Dec 31 t AND (no end date OR ended after t)
def active_stock(df: pd.DataFrame, year: int) -> pd.Series:
    cutoff = pd.Timestamp(f"{year}-12-31")
    mask = (df["StartDate"] <= cutoff) & (
        df["EndDate"].isna() | (df["EndDate"] > cutoff)
    )
    return df.loc[mask].groupby("KommuneCode").size()

stock_records = []
for y in years:
    s = active_stock(units, y)
    for kcode, n in s.items():
        stock_records.append({"KommuneCode": kcode, "year": y, "active": int(n)})
stock = pd.DataFrame(stock_records)

# Build complete panel: every (kommune, year) combo, filled with zeros
all_kcodes = sorted({str(x) for x in units["KommuneCode"].dropna()} | {str(x) for x in stats["KommuneCode"].dropna()})
panel = pd.MultiIndex.from_product([all_kcodes, years], names=["KommuneCode", "year"]).to_frame(index=False)
panel = (panel
    .merge(opens, on=["KommuneCode", "year"], how="left")
    .merge(closes, on=["KommuneCode", "year"], how="left")
    .merge(stock, on=["KommuneCode", "year"], how="left")
)
panel[["openings", "closings", "active"]] = panel[["openings", "closings", "active"]].fillna(0).astype(int)
panel["net_change"] = panel["openings"] - panel["closings"]

# Attach kommune name (use most recent name from the units file as authoritative)
name_lookup = (units.dropna(subset=["KommuneName"])
                    .drop_duplicates("KommuneCode")
                    .set_index("KommuneCode")["KommuneName"])
panel["KommuneName"] = panel["KommuneCode"].map(name_lookup)
# Fall back to stats names if missing
stats_names = stats.drop_duplicates("KommuneCode").set_index("KommuneCode")["municipality_name"]
panel["KommuneName"] = panel["KommuneName"].fillna(panel["KommuneCode"].map(stats_names))

print(f"  panel shape: {panel.shape}  (expected {len(all_kcodes)} × {len(years)} = {len(all_kcodes)*len(years)})")

# ----------------------------------------------------------------------------
# 3. CHART 1 — National time series
# ----------------------------------------------------------------------------
print("\n[1/4] National time series...")
national = panel.groupby("year").agg(
    openings=("openings", "sum"),
    closings=("closings", "sum"),
    net_change=("net_change", "sum"),
).reset_index()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=national["year"], y=national["openings"], name="Openings",
                          mode="lines+markers", line=dict(color="#2ca02c", width=3)))
fig1.add_trace(go.Scatter(x=national["year"], y=national["closings"], name="Closings",
                          mode="lines+markers", line=dict(color="#d62728", width=3)))
fig1.add_trace(go.Bar(x=national["year"], y=national["net_change"], name="Net change",
                      marker_color=["#2ca02c" if v >= 0 else "#d62728" for v in national["net_change"]],
                      opacity=0.35, yaxis="y2"))

# Annotate macro events
fig1.add_vrect(x0=2008.5, x1=2009.5, fillcolor="gray", opacity=0.1, line_width=0,
               annotation_text="Financial crisis", annotation_position="top left")
fig1.add_vrect(x0=2019.5, x1=2020.5, fillcolor="gray", opacity=0.1, line_width=0,
               annotation_text="COVID", annotation_position="top left")

fig1.update_layout(
    title="Production unit openings and closings, Denmark 2008–2024",
    xaxis_title="Year",
    yaxis=dict(title="Openings / Closings (units per year)"),
    yaxis2=dict(title="Net change", overlaying="y", side="right", showgrid=False),
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig1.write_html(OUT_DIR / "national_timeseries.html", include_plotlyjs="cdn", full_html=True)

# ----------------------------------------------------------------------------
# 4. CHART 2 — Active stock
# ----------------------------------------------------------------------------
print("[2/4] Active stock over time...")
region_lookup = (units.dropna(subset=["RegionName"])
                      .drop_duplicates("KommuneCode")
                      .set_index("KommuneCode")["RegionName"])
panel["RegionName"] = panel["KommuneCode"].map(region_lookup)

stock_national = panel.groupby("year")["active"].sum().reset_index()
stock_region = panel.dropna(subset=["RegionName"]).groupby(["year", "RegionName"])["active"].sum().reset_index()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=stock_national["year"], y=stock_national["active"],
                          name="Denmark (total)", mode="lines+markers",
                          line=dict(color="black", width=3)))
for region, grp in stock_region.groupby("RegionName"):
    fig2.add_trace(go.Scatter(x=grp["year"], y=grp["active"], name=region,
                              mode="lines+markers", line=dict(width=1.5), visible="legendonly"))
fig2.update_layout(
    title="Active production unit stock, year-end (toggle regions in legend)",
    xaxis_title="Year", yaxis_title="Active units",
    template="plotly_white", hovermode="x unified",
)
fig2.write_html(OUT_DIR / "active_stock.html", include_plotlyjs="cdn", full_html=True)

# ----------------------------------------------------------------------------
# 5. CHART 3 — Choropleth: cumulative net change, normalized
# ----------------------------------------------------------------------------
print("[3/4] Choropleth map...")
print("  fetching GeoJSON from dataforsyningen.dk...")
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson",
    timeout=30,
).json()

# Cumulative net change per kommune
cum = panel.groupby("KommuneCode").agg(
    openings_total=("openings", "sum"),
    closings_total=("closings", "sum"),
    net_total=("net_change", "sum"),
    active_2008=("active", "first"),
    active_2024=("active", "last"),
).reset_index()
cum["KommuneName"] = cum["KommuneCode"].map(name_lookup).fillna(cum["KommuneCode"].map(stats_names))

# Normalize by 2008 active stock (relative change). Guard against zero.
cum["pct_change"] = np.where(
    cum["active_2008"] > 0,
    100 * cum["net_total"] / cum["active_2008"],
    np.nan,
)

# Diverging scale centered on 0
abs_max = np.nanpercentile(np.abs(cum["pct_change"]), 95)  # clip to 95th pct so outliers don't blow out the scale

fig3 = px.choropleth_map(
    cum,
    geojson=geojson,
    locations="KommuneCode",
    featureidkey="properties.kode",
    color="pct_change",
    color_continuous_scale="RdYlGn",
    range_color=(-abs_max, abs_max),
    map_style="carto-positron",
    zoom=5.8, center={"lat": 56.0, "lon": 10.5},
    opacity=0.75,
    hover_name="KommuneName",
    hover_data={
        "KommuneCode": False,
        "openings_total": ":,",
        "closings_total": ":,",
        "net_total": ":+,",
        "pct_change": ":+.1f",
        "active_2008": ":,",
    },
    labels={"pct_change": "Net change (% of 2008 stock)"},
)
fig3.update_layout(
    title="Cumulative net change in production units, 2008–2024<br>"
          "<sub>Net change = openings − closings, expressed as % of 2008 active stock. "
          "Color clipped at ±95th pct for legibility.</sub>",
    margin=dict(l=0, r=0, t=80, b=0),
)
fig3.write_html(OUT_DIR / "choropleth_net_change.html", include_plotlyjs="cdn", full_html=True)

# ----------------------------------------------------------------------------
# 6. CHART 4 — Top movers ranking
# ----------------------------------------------------------------------------
print("[4/4] Top movers ranking...")
top_abs = cum.dropna(subset=["pct_change"]).nlargest(15, "net_total")
bot_abs = cum.dropna(subset=["pct_change"]).nsmallest(15, "net_total")
top_pct = cum.dropna(subset=["pct_change"]).nlargest(15, "pct_change")
bot_pct = cum.dropna(subset=["pct_change"]).nsmallest(15, "pct_change")

def make_bar(df, x_col, title, color):
    df = df.sort_values(x_col)
    return go.Bar(y=df["KommuneName"], x=df[x_col], orientation="h",
                  marker_color=color, name=title,
                  hovertemplate="%{y}: %{x:+,}<extra></extra>")

from plotly.subplots import make_subplots
fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Top 15 — absolute net gain", "Bottom 15 — absolute net loss",
        "Top 15 — % gain vs 2008 stock", "Bottom 15 — % loss vs 2008 stock",
    ),
    horizontal_spacing=0.20, vertical_spacing=0.12,
)
fig4.add_trace(make_bar(top_abs, "net_total", "abs gain", "#2ca02c"), row=1, col=1)
fig4.add_trace(make_bar(bot_abs, "net_total", "abs loss", "#d62728"), row=1, col=2)
fig4.add_trace(make_bar(top_pct, "pct_change", "% gain", "#2ca02c"), row=2, col=1)
fig4.add_trace(make_bar(bot_pct, "pct_change", "% loss", "#d62728"), row=2, col=2)
fig4.update_layout(
    title="Top movers, 2008–2024: which municipalities saw the biggest changes?",
    showlegend=False, template="plotly_white", height=900,
)
fig4.update_xaxes(title_text="Net units", row=1, col=1)
fig4.update_xaxes(title_text="Net units", row=1, col=2)
fig4.update_xaxes(title_text="% of 2008 stock", row=2, col=1)
fig4.update_xaxes(title_text="% of 2008 stock", row=2, col=2)
fig4.write_html(OUT_DIR / "top_movers.html", include_plotlyjs="cdn", full_html=True)

# ----------------------------------------------------------------------------
# Also dump the panel as CSV for use in the explanatory phase
# ----------------------------------------------------------------------------
panel_with_stats = panel.merge(
    stats[["KommuneCode", "year", "population", "average_income"]],
    on=["KommuneCode", "year"], how="left",
)
panel_with_stats.to_csv(OUT_DIR / "kommune_year_panel.csv", index=False)
print(f"\nPanel written: {OUT_DIR/'kommune_year_panel.csv'}  ({len(panel_with_stats):,} rows)")
print(f"All charts saved in: {OUT_DIR.resolve()}")


Loading data...
  dropped 582 short-lived registration artifacts (<30 days)
  dropped 18,477 units based on company name filters
  6,546 units, 99 municipalities in stats
  panel shape: (1683, 7)  (expected 99 × 17 = 1683)

[1/4] National time series...
[2/4] Active stock over time...
[3/4] Choropleth map...
  fetching GeoJSON from dataforsyningen.dk...
[4/4] Top movers ranking...

Panel written: outputs/kommune_year_panel.csv  (1,683 rows)
All charts saved in: /home/edgar/MSc_HCAI_Py/SocialDataFinalProject/notebooks/outputs


In [3]:
"""Extract top and bottom mover municipalities for the narrative.

Reads outputs/kommune_year_panel.csv (produced by eda.py) and prints:
  - Top 10 kommuner by net change % (positive)
  - Bottom 10 kommuner by net change % (negative)
  - Their actual population and income % changes for context
"""
import numpy as np
import pandas as pd

panel = pd.read_csv("outputs/kommune_year_panel.csv", dtype={"KommuneCode": str})
panel["KommuneCode"] = panel["KommuneCode"].str.zfill(4)

YEAR_START, YEAR_END = 2008, 2024
start = panel[panel["year"] == YEAR_START].set_index("KommuneCode")
end = panel[panel["year"] == YEAR_END].set_index("KommuneCode")

totals = panel.groupby("KommuneCode")[["openings", "closings", "net_change"]].sum()

df = pd.DataFrame({
    "KommuneName":        start["KommuneName"],
    "active_2008":        start["active"],
    "active_2024":        end["active"],
    "population_2008":    start["population"],
    "population_2024":    end["population"],
    "income_2008":        start["average_income"],
    "income_2024":        end["average_income"],
    "openings":           totals["openings"],
    "closings":           totals["closings"],
    "net_total":          totals["net_change"],
})

df["net_pct"] = np.where(df["active_2008"] > 0,
                         100 * df["net_total"] / df["active_2008"], np.nan)
df["pop_pct"] = 100 * (df["population_2024"] - df["population_2008"]) / df["population_2008"]
df["income_pct"] = 100 * (df["income_2024"] - df["income_2008"]) / df["income_2008"]

# Drop kommuner with insufficient baseline (active_2008 == 0 → net_pct is nan)
df_valid = df.dropna(subset=["net_pct"]).copy()

cols = ["KommuneName", "active_2008", "active_2024", "openings", "closings",
        "net_total", "net_pct", "pop_pct", "income_pct"]

print(f"\n=== Top 10 by net change % (biggest gains) ===")
print(df_valid.nlargest(10, "net_pct")[cols].to_string(
    index=False,
    formatters={"net_pct": "{:+.1f}".format, "pop_pct": "{:+.1f}".format,
                "income_pct": "{:+.1f}".format}))

print(f"\n=== Bottom 10 by net change % (biggest losses) ===")
print(df_valid.nsmallest(10, "net_pct")[cols].to_string(
    index=False,
    formatters={"net_pct": "{:+.1f}".format, "pop_pct": "{:+.1f}".format,
                "income_pct": "{:+.1f}".format}))

# Quadrant analysis — units down AND population down, etc.
print(f"\n=== Quadrant counts (n = {len(df_valid)}) ===")
qq = pd.crosstab(df_valid["net_pct"] >= 0, df_valid["pop_pct"] >= 0,
                 rownames=["units ≥ 0"], colnames=["pop ≥ 0"])
print(qq)

from scipy import stats
sub = df_valid.dropna(subset=["net_pct", "income_pct"])
r_inc, p_inc = stats.pearsonr(sub["net_pct"], sub["income_pct"])
print(f"Net change % vs income %: r = {r_inc:+.3f}, p = {p_inc:.3g}  (n={len(sub)})")

# Save the full table for reuse
df_valid.sort_values("net_pct", ascending=False).to_csv(
    "outputs/kommune_movers.csv", index=False)
print(f"\nFull table → outputs/kommune_movers.csv ({len(df_valid)} rows)")


=== Top 10 by net change % (biggest gains) ===
  KommuneName  active_2008  active_2024  openings  closings  net_total net_pct pop_pct income_pct
   Vallensbæk            1            7         8         2          6  +600.0   +43.6      +10.5
   Kalundborg           12           63        61         9         52  +433.3    -2.9      +28.2
      Brøndby            3           11        21        13          8  +266.7   +15.5      +11.5
      Billund           14           47        40         9         31  +221.4    +3.6      +29.0
     Gladsaxe           19           60        75        33         42  +221.1   +12.8      +25.9
     Hillerød           16           48        50        16         34  +212.5   +16.9      +23.2
     Ballerup           20           56        69        31         38  +190.0    +8.7      +18.5
Høje-Taastrup           20           47        56        24         32  +160.0   +22.0      +12.2
       Dragør            2            5         6         3          3

In [8]:
"""
Spatial analysis additions to the EDA. Reuses the kommune_year_panel.csv
produced by eda.py and adds:

  5. scatter_units_vs_population.html   — % change in active stock vs % change in population
  6. folium_bivariate_map.html          — interactive layered map: units + population change
  7. folium_animated_yearly.html        — TimeSliderChoropleth: cumulative net change by year

Run eda.py first to produce outputs/kommune_year_panel.csv.
"""
from __future__ import annotations

import json
from pathlib import Path

import branca.colormap as cm
import folium
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from folium.plugins import TimeSliderChoropleth

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
PANEL_PATH = Path("outputs/kommune_year_panel.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

YEAR_START, YEAR_END = 2008, 2024

# ----------------------------------------------------------------------------
# Load
# ----------------------------------------------------------------------------
panel = pd.read_csv(PANEL_PATH, dtype={"KommuneCode": str})
panel["KommuneCode"] = panel["KommuneCode"].str.zfill(4)
print(f"Panel: {len(panel):,} rows, {panel['KommuneCode'].nunique()} kommuner, "
      f"{panel['year'].min()}–{panel['year'].max()}")

print("Fetching GeoJSON...")
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson", timeout=30
).json()

# ----------------------------------------------------------------------------
# Compute one-row-per-kommune summary
# ----------------------------------------------------------------------------
def first_last(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Return first-year and last-year values per kommune for a given column."""
    first = df.sort_values("year").groupby("KommuneCode").first()[col].rename(f"{col}_start")
    last = df.sort_values("year").groupby("KommuneCode").last()[col].rename(f"{col}_end")
    return pd.concat([first, last], axis=1)

agg = pd.concat([
    first_last(panel, "active"),
    first_last(panel, "population"),
    first_last(panel, "average_income"),
    panel.groupby("KommuneCode")[["openings", "closings", "net_change"]].sum(),
    panel.groupby("KommuneCode")["KommuneName"].first(),
], axis=1).reset_index()

# Percent changes (guard against zero-stock kommuner)
agg["active_pct"] = np.where(
    agg["active_start"] > 0,
    100 * (agg["active_end"] - agg["active_start"]) / agg["active_start"],
    np.nan,
)
agg["pop_pct"] = 100 * (agg["population_end"] - agg["population_start"]) / agg["population_start"]
agg["income_pct"] = 100 * (agg["average_income_end"] - agg["average_income_start"]) / agg["average_income_start"]

print(f"Summary rows: {len(agg)}")

# ----------------------------------------------------------------------------
# CHART 5 — Scatter: net unit change vs population change
# ----------------------------------------------------------------------------
print("\n[5] Scatter: units vs population...")
scatter_df = agg.dropna(subset=["active_pct", "pop_pct", "population_start"]).copy()

# Correlation + simple OLS line
from scipy import stats as sp_stats
r, p = sp_stats.pearsonr(scatter_df["active_pct"], scatter_df["pop_pct"])
slope, intercept = np.polyfit(scatter_df["active_pct"], scatter_df["pop_pct"], 1)
x_line = np.linspace(scatter_df["active_pct"].min(), scatter_df["active_pct"].max(), 100)
y_line = slope * x_line + intercept

fig5 = px.scatter(
    scatter_df,
    x="active_pct", y="pop_pct",
    size="population_start", color="KommuneName",
    hover_name="KommuneName",
    hover_data={
        "active_start": ":,", "active_end": ":,",
        "population_start": ":,", "population_end": ":,",
        "net_change": ":+,",
        "active_pct": ":+.1f", "pop_pct": ":+.1f",
        "KommuneName": False,
    },
    labels={
        "active_pct": "Active production units, % change 2008→2024",
        "pop_pct": "Population, % change 2008→2024",
    },
    size_max=40,
)
fig5.add_trace(go.Scatter(
    x=x_line, y=y_line, mode="lines",
    line=dict(color="black", dash="dash", width=2),
    name=f"OLS: y = {slope:.2f}x + {intercept:.2f}",
    hoverinfo="skip",
))
fig5.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)
fig5.add_vline(x=0, line_dash="dot", line_color="gray", opacity=0.5)
fig5.update_layout(
    title=(f"Do municipalities with more production units gain population?<br>"
           f"<sub>Pearson r = {r:.3f}, p = {p:.3g}, n = {len(scatter_df)}. "
           f"Dot size = 2008 population.</sub>"),
    template="plotly_white", showlegend=False, height=650,
)
# Quadrant labels
xr = scatter_df["active_pct"].abs().max() * 0.9
yr = scatter_df["pop_pct"].abs().max() * 0.9
for x, y, txt in [(xr, yr, "units ↑ pop ↑"), (-xr, yr, "units ↓ pop ↑"),
                  (-xr, -yr, "units ↓ pop ↓"), (xr, -yr, "units ↑ pop ↓")]:
    fig5.add_annotation(x=x, y=y, text=f"<i>{txt}</i>", showarrow=False,
                        font=dict(size=11, color="gray"), opacity=0.6)
fig5.write_html(OUT_DIR / "scatter_units_vs_population.html",
                include_plotlyjs="cdn", full_html=True)
print(f"  r = {r:+.3f}, p = {p:.3g}, slope = {slope:+.3f}")

# ----------------------------------------------------------------------------
# CHART 6 — Folium bivariate-style map: units (choropleth) + population (markers)
# ----------------------------------------------------------------------------
print("\n[6] Folium layered map...")

# Build lookup keyed by kommune code (as in GeoJSON properties.kode)
agg_idx = agg.set_index("KommuneCode")

m = folium.Map(location=[56.0, 10.5], zoom_start=7, tiles="cartodbpositron",
               control_scale=True)

# Layer A: choropleth of % change in active units
units_cmap = cm.LinearColormap(
    colors=["#b2182b", "#ef8a62", "#fddbc7", "#f7f7f7", "#d1e5f0", "#67a9cf", "#2166ac"],
    vmin=-max(abs(agg["active_pct"].dropna().quantile(0.05)),
              abs(agg["active_pct"].dropna().quantile(0.95))),
    vmax= max(abs(agg["active_pct"].dropna().quantile(0.05)),
              abs(agg["active_pct"].dropna().quantile(0.95))),
    caption="Active production units: % change 2008–2024",
)

def style_for_units(feature):
    code = feature["properties"]["kode"]
    val = agg_idx["active_pct"].get(code, np.nan)
    color = "#cccccc" if pd.isna(val) else units_cmap(val)
    return {"fillColor": color, "color": "white", "weight": 0.5, "fillOpacity": 0.75}

def make_tooltip_fields():
    return ["navn"]

units_layer = folium.FeatureGroup(name="Units: % change", show=True)
folium.GeoJson(
    geojson,
    style_function=style_for_units,
    tooltip=folium.GeoJsonTooltip(fields=["navn"], aliases=["Kommune:"]),
    popup=folium.GeoJsonPopup(
        fields=["navn", "kode"],
        aliases=["Kommune:", "Code:"],
    ),
    name="Units choropleth",
).add_to(units_layer)
units_layer.add_to(m)
units_cmap.add_to(m)

# Layer B: bubbles sized by absolute population change, colored by sign
pop_layer = folium.FeatureGroup(name="Population change (bubbles)", show=True)

# Get centroid of each kommune polygon for bubble placement
from shapely.geometry import shape
for feat in geojson["features"]:
    code = feat["properties"]["kode"]
    if code not in agg_idx.index:
        continue
    row = agg_idx.loc[code]
    if pd.isna(row.get("pop_pct")):
        continue
    centroid = shape(feat["geometry"]).centroid
    pop_delta = row["population_end"] - row["population_start"]
    radius = max(3, min(25, np.sqrt(abs(pop_delta)) / 8))
    color = "#1a9850" if pop_delta >= 0 else "#d73027"
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=radius,
        color=color, fill=True, fill_color=color, fill_opacity=0.55, weight=1,
        popup=folium.Popup(
            f"<b>{row['KommuneName']}</b><br>"
            f"Units: {row['active_start']:,.0f} → {row['active_end']:,.0f} "
            f"({row['active_pct']:+.1f}%)<br>"
            f"Population: {row['population_start']:,.0f} → {row['population_end']:,.0f} "
            f"({row['pop_pct']:+.1f}%)<br>"
            f"Avg income: {row['income_pct']:+.1f}%",
            max_width=300,
        ),
        tooltip=f"{row['KommuneName']}: pop {row['pop_pct']:+.1f}%",
    ).add_to(pop_layer)
pop_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element("""
<div style="position: fixed; bottom: 20px; left: 20px; background: white; padding: 10px;
            border: 1px solid #888; border-radius: 4px; font-size: 12px; z-index: 9999;
            max-width: 260px;">
  <b>Bubble legend</b><br>
  Bubble = absolute population change.<br>
  <span style="color:#1a9850;">●</span> green: population grew<br>
  <span style="color:#d73027;">●</span> red: population shrank<br>
  Choropleth = % change in active production units.
</div>
"""))

m.save(str(OUT_DIR / "folium_bivariate_map.html"))

# ----------------------------------------------------------------------------
# CHART 7 — Folium animated yearly map (TimeSliderChoropleth)
# ----------------------------------------------------------------------------
print("\n[7] Folium animated map (cumulative net change by year)...")

# Compute cumulative net change per kommune-year
panel_sorted = panel.sort_values(["KommuneCode", "year"]).copy()
panel_sorted["cum_net"] = panel_sorted.groupby("KommuneCode")["net_change"].cumsum()

# Normalize by 2008 active stock so kommuner are comparable
stock_2008 = panel_sorted[panel_sorted["year"] == YEAR_START].set_index("KommuneCode")["active"]
panel_sorted["cum_net_pct"] = panel_sorted.apply(
    lambda r: 100 * r["cum_net"] / stock_2008.get(r["KommuneCode"], np.nan)
    if stock_2008.get(r["KommuneCode"], 0) > 0 else np.nan,
    axis=1,
)

# Build colormap with symmetric range clipped at 95th pct
abs_max = np.nanpercentile(np.abs(panel_sorted["cum_net_pct"]), 95)
anim_cmap = cm.LinearColormap(
    colors=["#b2182b", "#ef8a62", "#fddbc7", "#f7f7f7", "#d1e5f0", "#67a9cf", "#2166ac"],
    vmin=-abs_max, vmax=abs_max,
    caption="Cumulative net change in production units (% of 2008 stock)",
)

# TimeSliderChoropleth expects: {feature_id: {epoch_seconds_str: {color, opacity}}}
styledata = {}
for code, grp in panel_sorted.groupby("KommuneCode"):
    entries = {}
    for _, row in grp.iterrows():
        # Use Jan 1 of the year as the timestamp
        ts = str(int(pd.Timestamp(f"{int(row['year'])}-01-01").timestamp()))
        val = row["cum_net_pct"]
        color = "#cccccc" if pd.isna(val) else anim_cmap(np.clip(val, -abs_max, abs_max))
        entries[ts] = {"color": color, "opacity": 0.75}
    styledata[code] = entries

# Attach kode as feature id for TimeSliderChoropleth to find it
for feat in geojson["features"]:
    feat["id"] = feat["properties"]["kode"]

m_anim = folium.Map(location=[56.0, 10.5], zoom_start=7, tiles="cartodbpositron")
TimeSliderChoropleth(
    data=json.dumps(geojson),
    styledict=styledata,
).add_to(m_anim)
anim_cmap.add_to(m_anim)

m_anim.get_root().html.add_child(folium.Element("""
<div style="position: fixed; top: 80px; right: 20px; background: white; padding: 10px;
            border: 1px solid #888; border-radius: 4px; font-size: 12px; z-index: 9999;
            max-width: 280px;">
  <b>Cumulative net change since 2008</b><br>
  Each frame shows openings − closings accumulated from 2008 to that year,
  normalized by 2008 stock.<br>
  Drag the slider at the bottom to step through years.<br>
  <span style="color:#2166ac;">■</span> blue: gain &nbsp;
  <span style="color:#b2182b;">■</span> red: loss &nbsp;
  <span style="color:#cccccc;">■</span> no data
</div>
"""))

m_anim.save(str(OUT_DIR / "folium_animated_yearly.html"))

print(f"\nAll outputs saved to: {OUT_DIR.resolve()}")

Panel: 1,683 rows, 99 kommuner, 2008–2024
Fetching GeoJSON...
Summary rows: 99

[5] Scatter: units vs population...
  r = +0.386, p = 7.99e-05, slope = +0.043

[6] Folium layered map...

[7] Folium animated map (cumulative net change by year)...

All outputs saved to: /home/edgar/MSc_HCAI_Py/SocialDataFinalProject/notebooks/outputs


In [9]:
"""
Two pairs of maps (folium + plotly), one for population change and one for
average income change 2008→2024, each with markers for production units
opened or closed in that window.

Marker rules (per user spec):
  - Green = opened ≥ 2008-01-01 AND still open (no end date)
  - Red   = opened ≥ 2008-01-01 AND closed ≥ 2008-01-01
  - Units that opened before 2008 are NOT shown as markers

Hover on marker: Start date, End date, Company name, VAT, Kommune name
Hover on kommune: Name, % change in pop/income, Net total (opens-closes), Net change %

Outputs in ./outputs/:
  - map_population_folium.html
  - map_population_plotly.html
  - map_income_folium.html
  - map_income_plotly.html
"""
from __future__ import annotations

import json
from pathlib import Path

import branca.colormap as cm
import folium
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from folium.plugins import MarkerCluster

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
UNITS_PATH = Path("../data/CVR_API/cvr_production_units_geocoded.csv")
STATS_PATH = Path("../data/geograph/statsbank_combined.csv")
PANEL_PATH = Path("outputs/kommune_year_panel.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

YEAR_START, YEAR_END = 2008, 2024
PERIOD_START = pd.Timestamp(f"{YEAR_START}-01-01")
MIN_DURATION_DAYS = 30

EXCLUDE_NAME_PATTERNS = [
    r"\bKommune\b", r"\bRegion\b", r"\bSalling\b", r"\b365discount\b",
    r"\bCOOP\b", r"\bPOST", r"\bpoliti\b", r"\bCOMPASS\b",
]

# ----------------------------------------------------------------------------
# 1. Load units (re-apply the cleaning from eda.py)
# ----------------------------------------------------------------------------
def load_units() -> pd.DataFrame:
    df = pd.read_csv(UNITS_PATH, dtype={"KommuneCode": str}, low_memory=False)
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
    df["KommuneCode"] = df["KommuneCode"].str.zfill(4)

    # Name-based exclusions
    name = df["CompanyName"].fillna("")
    mask = pd.Series(False, index=df.index)
    for pat in EXCLUDE_NAME_PATTERNS:
        mask |= name.str.contains(pat, case=False, regex=True, na=False)
    df = df.loc[~mask].copy()

    # Date parsing
    df["StartDate"] = pd.to_datetime(
        df["UnitStartdate"].str.replace(" ", "", regex=False),
        format="%d/%m-%Y", errors="coerce",
    )
    df["EndDate"] = pd.to_datetime(df["UnitEnddate"], errors="coerce")

    # Drop registration artifacts
    dur = (df["EndDate"] - df["StartDate"]).dt.days
    df = df.loc[~(dur.notna() & (dur < MIN_DURATION_DAYS))].copy()
    return df


print("Loading units...")
units = load_units()
print(f"  {len(units):,} units after filtering")

# ----------------------------------------------------------------------------
# 2. Classify markers per the user's rule set
# ----------------------------------------------------------------------------
# Marker classification (per user spec, updated):
#   Green: opened on/after 2008-01-01 AND still open
#   Red:   closed on/after 2008-01-01 (regardless of when it opened, so that a
#          unit opened in 2005 and closed in 2010 still appears as a red marker)
opened_in_period = units["StartDate"] >= PERIOD_START
still_open = units["EndDate"].isna()
closed_in_period = units["EndDate"] >= PERIOD_START

green_units = units[opened_in_period & still_open].copy()
red_units = units[closed_in_period].copy()

# Drop missing coordinates
green_units = green_units.dropna(subset=["Lat", "Lon"])
red_units = red_units.dropna(subset=["Lat", "Lon"])
print(f"  green (opened in period, still open): {len(green_units):,}")
print(f"  red   (closed in period, any open date): {len(red_units):,}")

# ----------------------------------------------------------------------------
# 3. Compute kommune-level metrics for both maps
# ----------------------------------------------------------------------------
print("Computing kommune metrics...")
panel = pd.read_csv(PANEL_PATH, dtype={"KommuneCode": str})
panel["KommuneCode"] = panel["KommuneCode"].str.zfill(4)

start_row = panel[panel["year"] == YEAR_START].set_index("KommuneCode")
end_row = panel[panel["year"] == YEAR_END].set_index("KommuneCode")

metrics = pd.DataFrame(index=sorted(set(start_row.index) | set(end_row.index)))
metrics.index.name = "KommuneCode"
metrics["KommuneName"] = start_row["KommuneName"]
metrics["population_start"] = start_row["population"]
metrics["population_end"] = end_row["population"]
metrics["income_start"] = start_row["average_income"]
metrics["income_end"] = end_row["average_income"]

# % changes
metrics["pop_pct"] = 100 * (metrics["population_end"] - metrics["population_start"]) / metrics["population_start"]
metrics["income_pct"] = 100 * (metrics["income_end"] - metrics["income_start"]) / metrics["income_start"]

# Net total (openings - closings 2008–2024) and net change as % of 2008 stock
totals = panel.groupby("KommuneCode")[["openings", "closings", "net_change"]].sum()
metrics["openings_total"] = totals["openings"]
metrics["closings_total"] = totals["closings"]
metrics["net_total"] = totals["net_change"]
active_2008 = start_row["active"]
metrics["net_pct"] = np.where(
    active_2008.reindex(metrics.index) > 0,
    100 * metrics["net_total"] / active_2008.reindex(metrics.index),
    np.nan,
)
metrics = metrics.reset_index()

# ----------------------------------------------------------------------------
# 4. GeoJSON
# ----------------------------------------------------------------------------
print("Fetching GeoJSON...")
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson", timeout=30
).json()

# Inject metrics into each feature's properties so Plotly/Folium hover can use them
metrics_by_code = metrics.set_index("KommuneCode").to_dict("index")
for feat in geojson["features"]:
    code = feat["properties"]["kode"]
    m = metrics_by_code.get(code, {})
    feat["properties"]["pop_pct"] = m.get("pop_pct")
    feat["properties"]["income_pct"] = m.get("income_pct")
    feat["properties"]["net_total"] = m.get("net_total")
    feat["properties"]["net_pct"] = m.get("net_pct")
    feat["properties"]["kommune_name"] = m.get("KommuneName") or feat["properties"].get("navn")
    feat["id"] = code  # for Plotly choropleth matching

# ----------------------------------------------------------------------------
# 5. Helpers
# ----------------------------------------------------------------------------
def fmt_date(d) -> str:
    return d.strftime("%Y-%m-%d") if pd.notna(d) else "still open"

def build_folium_map(value_col: str, value_label: str, output_name: str):
    """Folium choropleth + marker layers."""
    m_data = metrics.dropna(subset=[value_col])
    vals = m_data[value_col]
    # Symmetric range around zero if data crosses zero, else min-max
    if (vals.min() < 0) and (vals.max() > 0):
        bound = max(abs(vals.quantile(0.05)), abs(vals.quantile(0.95)))
        vmin, vmax = -bound, bound
        colors = ["#b2182b", "#ef8a62", "#fddbc7", "#f7f7f7", "#d1e5f0", "#67a9cf", "#2166ac"]
    else:
        vmin, vmax = vals.quantile(0.05), vals.quantile(0.95)
        colors = ["#fff5f0", "#fdbb84", "#e34a33", "#b30000"] if vmin >= 0 else \
                 ["#08306b", "#4292c6", "#deebf7", "#fff5f0"]
    cmap = cm.LinearColormap(colors=colors, vmin=vmin, vmax=vmax, caption=value_label)

    fmap = folium.Map(location=[56.0, 10.5], zoom_start=7,
                      tiles="cartodbpositron", control_scale=True)

    def style_fn(feat):
        v = feat["properties"].get(value_col)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            color = "#cccccc"
        else:
            color = cmap(np.clip(v, vmin, vmax))
        return {"fillColor": color, "color": "white", "weight": 0.6, "fillOpacity": 0.75}

    # Choropleth layer with rich tooltip
    folium.GeoJson(
        geojson,
        style_function=style_fn,
        name="Municipalities",
        tooltip=folium.GeoJsonTooltip(
            fields=["kommune_name", value_col, "net_total", "net_pct"],
            aliases=["Kommune:", f"{value_label}:", "Net total (units):", "Net change (%):"],
            localize=True,
            sticky=False,
            labels=True,
            style="background-color: white; font-family: sans-serif; font-size: 12px;",
        ),
    ).add_to(fmap)

    # Marker layers — clustered for performance
    green_cluster = MarkerCluster(name=f"Opened in {YEAR_START}–{YEAR_END} (still open)",
                                  show=True).add_to(fmap)
    red_cluster = MarkerCluster(name=f"Closed in {YEAR_START}–{YEAR_END}",
                                show=True).add_to(fmap)

    def popup_html(row):
        return folium.Popup(
            f"<div style='font-size:12px;'>"
            f"<b>{row['CompanyName']}</b><br>"
            f"VAT: {row['CompanyVat']}<br>"
            f"Kommune: {row['KommuneName']}<br>"
            f"Opened: {fmt_date(row['StartDate'])}<br>"
            f"Closed: {fmt_date(row['EndDate'])}"
            f"</div>",
            max_width=280,
        )

    for _, r in green_units.iterrows():
        folium.CircleMarker(
            location=[r["Lat"], r["Lon"]], radius=3,
            color="#1a9850", fill=True, fill_color="#1a9850", fill_opacity=0.7, weight=1,
            popup=popup_html(r),
            tooltip=f"{r['CompanyName']} (open)",
        ).add_to(green_cluster)

    for _, r in red_units.iterrows():
        folium.CircleMarker(
            location=[r["Lat"], r["Lon"]], radius=3,
            color="#d73027", fill=True, fill_color="#d73027", fill_opacity=0.7, weight=1,
            popup=popup_html(r),
            tooltip=f"{r['CompanyName']} (closed)",
        ).add_to(red_cluster)

    cmap.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.save(str(OUT_DIR / output_name))
    print(f"  wrote {output_name}")


def build_plotly_map(value_col: str, value_label: str, output_name: str):
    """Plotly choropleth + scattermap markers, all in one figure."""
    m_data = metrics.dropna(subset=[value_col]).copy()
    vals = m_data[value_col]
    if (vals.min() < 0) and (vals.max() > 0):
        bound = max(abs(vals.quantile(0.05)), abs(vals.quantile(0.95)))
        rng = (-bound, bound)
        scale = "RdBu"
    else:
        rng = (vals.quantile(0.05), vals.quantile(0.95))
        scale = "Reds" if vals.min() >= 0 else "Blues_r"

    fig = px.choropleth_map(
        m_data,
        geojson=geojson,
        locations="KommuneCode",
        featureidkey="properties.kode",
        color=value_col,
        color_continuous_scale=scale,
        range_color=rng,
        map_style="carto-positron",
        zoom=5.8, center={"lat": 56.0, "lon": 10.5},
        opacity=0.70,
        hover_name="KommuneName",
        hover_data={
            "KommuneCode": False,
            value_col: ":+.2f",
            "net_total": ":+,",
            "net_pct": ":+.2f",
        },
        labels={value_col: value_label, "net_total": "Net total (units)",
                "net_pct": "Net change (%)"},
    )

    # Add unit markers
    def hover_text(df):
        return [
            f"<b>{r.CompanyName}</b><br>"
            f"VAT: {r.CompanyVat}<br>"
            f"Kommune: {r.KommuneName}<br>"
            f"Opened: {fmt_date(r.StartDate)}<br>"
            f"Closed: {fmt_date(r.EndDate)}"
            for r in df.itertuples()
        ]

    fig.add_trace(go.Scattermap(
        lat=green_units["Lat"], lon=green_units["Lon"],
        mode="markers",
        marker=dict(size=6, color="#1a9850", opacity=0.7),
        name=f"Opened {YEAR_START}–{YEAR_END} (still open)",
        hovertext=hover_text(green_units),
        hoverinfo="text",
    ))
    fig.add_trace(go.Scattermap(
        lat=red_units["Lat"], lon=red_units["Lon"],
        mode="markers",
        marker=dict(size=6, color="#d73027", opacity=0.7),
        name=f"Closed {YEAR_START}–{YEAR_END}",
        hovertext=hover_text(red_units),
        hoverinfo="text",
    ))

    fig.update_layout(
        title=f"{value_label}, 2008–2024",
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01,
                    bgcolor="rgba(255,255,255,0.85)"),
        height=750,
    )
    fig.write_html(OUT_DIR / output_name, include_plotlyjs="cdn", full_html=True)
    print(f"  wrote {output_name}")


# ----------------------------------------------------------------------------
# 6. Build the four maps
# ----------------------------------------------------------------------------
print("\nMap 1: Population change")
build_folium_map("pop_pct", "Population change 2008–2024 (%)", "map_population_folium.html")
build_plotly_map("pop_pct", "Population change 2008–2024 (%)", "map_population_plotly.html")

print("\nMap 2: Average income change")
build_folium_map("income_pct", "Avg income change 2008–2024 (%)", "map_income_folium.html")
build_plotly_map("income_pct", "Avg income change 2008–2024 (%)", "map_income_plotly.html")

print(f"\nAll maps saved to {OUT_DIR.resolve()}")

Loading units...
  7,076 units after filtering
  green (opened in period, still open): 2,596
  red   (closed in period, any open date): 2,651
Computing kommune metrics...
Fetching GeoJSON...

Map 1: Population change
  wrote map_population_folium.html
  wrote map_population_plotly.html

Map 2: Average income change
  wrote map_income_folium.html
  wrote map_income_plotly.html

All maps saved to /home/edgar/MSc_HCAI_Py/SocialDataFinalProject/notebooks/outputs


In [10]:
"""
Two pairs of maps (folium + plotly), one for population change and one for
average income change 2008→2024. Each map shows:

  - Choropleth: % change in pop (Map 1) or income (Map 2), 2008→2024
  - One bubble per municipality at its polygon centroid, sized by |net change %|,
    colored green/red by sign (grey if zero), labeled with the signed value.

Hover on kommune (polygon): Name, % change in pop/income, Net total, Net change (%)
Bubble: no hover; value drawn inside.

Outputs in ./outputs/:
  - map_population_folium.html
  - map_population_plotly.html
  - map_income_folium.html
  - map_income_plotly.html
"""
from __future__ import annotations

import json
from pathlib import Path

import branca.colormap as cm
import folium
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from shapely.geometry import shape

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
PANEL_PATH = Path("outputs/kommune_year_panel.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

YEAR_START, YEAR_END = 2008, 2024

# Bubble sizing — radius in pixels (folium) and approx px (plotly).
# These are mapped from |net_pct| to a [min, max] radius linearly.
BUBBLE_MIN_PX = 8
BUBBLE_MAX_PX = 36

# ----------------------------------------------------------------------------
# 1. Kommune-level metrics
# ----------------------------------------------------------------------------
print("Loading panel...")
panel = pd.read_csv(PANEL_PATH, dtype={"KommuneCode": str})
panel["KommuneCode"] = panel["KommuneCode"].str.zfill(4)

start_row = panel[panel["year"] == YEAR_START].set_index("KommuneCode")
end_row = panel[panel["year"] == YEAR_END].set_index("KommuneCode")

metrics = pd.DataFrame(index=sorted(set(start_row.index) | set(end_row.index)))
metrics.index.name = "KommuneCode"
metrics["KommuneName"] = start_row["KommuneName"]
metrics["population_start"] = start_row["population"]
metrics["population_end"] = end_row["population"]
metrics["income_start"] = start_row["average_income"]
metrics["income_end"] = end_row["average_income"]

metrics["pop_pct"] = 100 * (metrics["population_end"] - metrics["population_start"]) / metrics["population_start"]
metrics["income_pct"] = 100 * (metrics["income_end"] - metrics["income_start"]) / metrics["income_start"]

totals = panel.groupby("KommuneCode")[["openings", "closings", "net_change"]].sum()
metrics["openings_total"] = totals["openings"]
metrics["closings_total"] = totals["closings"]
metrics["net_total"] = totals["net_change"]

active_2008 = start_row["active"]
metrics["net_pct"] = np.where(
    active_2008.reindex(metrics.index) > 0,
    100 * metrics["net_total"] / active_2008.reindex(metrics.index),
    np.nan,
)
metrics = metrics.reset_index()
print(f"  {len(metrics)} kommuner with metrics")

# ----------------------------------------------------------------------------
# 2. GeoJSON + centroids
# ----------------------------------------------------------------------------
print("Fetching GeoJSON...")
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson", timeout=30
).json()

# Centroid per kommune from polygon geometry
centroids = {}
for feat in geojson["features"]:
    code = feat["properties"]["kode"]
    c = shape(feat["geometry"]).centroid
    centroids[code] = (c.y, c.x)  # (lat, lon)
    feat["id"] = code

# Attach metrics to features (for choropleth hover)
metrics_by_code = metrics.set_index("KommuneCode").to_dict("index")
for feat in geojson["features"]:
    code = feat["properties"]["kode"]
    m = metrics_by_code.get(code, {})
    feat["properties"]["pop_pct"] = m.get("pop_pct")
    feat["properties"]["income_pct"] = m.get("income_pct")
    feat["properties"]["net_total"] = m.get("net_total")
    feat["properties"]["net_pct"] = m.get("net_pct")
    feat["properties"]["kommune_name"] = m.get("KommuneName") or feat["properties"].get("navn")

# ----------------------------------------------------------------------------
# 3. Bubble sizing helper
# ----------------------------------------------------------------------------
# Use the full range of |net_pct| across kommuner to scale radii linearly
abs_pct = metrics["net_pct"].abs().dropna()
PCT_MAX = float(abs_pct.max()) if len(abs_pct) > 0 else 1.0

def bubble_radius(net_pct: float) -> float:
    if pd.isna(net_pct):
        return BUBBLE_MIN_PX
    frac = min(abs(net_pct) / PCT_MAX, 1.0) if PCT_MAX > 0 else 0
    return BUBBLE_MIN_PX + frac * (BUBBLE_MAX_PX - BUBBLE_MIN_PX)

def bubble_color(net_pct: float) -> str:
    if pd.isna(net_pct) or abs(net_pct) < 1e-9:
        return "#888888"
    return "#1a9850" if net_pct > 0 else "#d73027"

def bubble_label(net_pct: float) -> str:
    if pd.isna(net_pct):
        return "n/a"
    return f"{net_pct:+.1f}%"

# ----------------------------------------------------------------------------
# 4. Folium map builder
# ----------------------------------------------------------------------------
def build_folium_map(value_col: str, value_label: str, output_name: str):
    m_data = metrics.dropna(subset=[value_col])
    vals = m_data[value_col]
    if (vals.min() < 0) and (vals.max() > 0):
        bound = max(abs(vals.quantile(0.05)), abs(vals.quantile(0.95)))
        vmin, vmax = -bound, bound
        colors = ["#b2182b", "#ef8a62", "#fddbc7", "#f7f7f7", "#d1e5f0", "#67a9cf", "#2166ac"]
    else:
        vmin, vmax = vals.quantile(0.05), vals.quantile(0.95)
        colors = ["#fff5f0", "#fdbb84", "#e34a33", "#b30000"] if vmin >= 0 else \
                 ["#08306b", "#4292c6", "#deebf7", "#fff5f0"]
    cmap = cm.LinearColormap(colors=colors, vmin=vmin, vmax=vmax, caption=value_label)

    fmap = folium.Map(location=[56.0, 10.5], zoom_start=7,
                      tiles="cartodbpositron", control_scale=True)

    def style_fn(feat):
        v = feat["properties"].get(value_col)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            color = "#cccccc"
        else:
            color = cmap(np.clip(v, vmin, vmax))
        return {"fillColor": color, "color": "white", "weight": 0.6, "fillOpacity": 0.75}

    # Choropleth with rich tooltip
    folium.GeoJson(
        geojson,
        style_function=style_fn,
        name="Municipalities",
        tooltip=folium.GeoJsonTooltip(
            fields=["kommune_name", value_col, "net_total", "net_pct"],
            aliases=["Kommune:", f"{value_label}:", "Net total (units):", "Net change (%):"],
            localize=True,
            sticky=False,
            labels=True,
            style="background-color: white; font-family: sans-serif; font-size: 12px;",
        ),
    ).add_to(fmap)

    # Bubble layer — one per kommune
    bubble_layer = folium.FeatureGroup(name="Net change % bubbles", show=True)
    for _, row in metrics.iterrows():
        code = row["KommuneCode"]
        if code not in centroids:
            continue
        lat, lon = centroids[code]
        r = bubble_radius(row["net_pct"])
        c = bubble_color(row["net_pct"])
        label = bubble_label(row["net_pct"])

        # The bubble itself, no interactivity (so it doesn't intercept the polygon tooltip)
        folium.CircleMarker(
            location=[lat, lon], radius=r,
            color=c, weight=1, fill=True, fill_color=c, fill_opacity=0.6,
            interactive=False,
        ).add_to(bubble_layer)

        # Text label centered on the bubble. Font size scales mildly with radius.
        font_px = max(9, min(14, int(r * 0.5)))
        icon_html = (
            f"<div style='font-size:{font_px}px; font-weight:600; color:white; "
            f"text-align:center; text-shadow: 0 0 2px rgba(0,0,0,0.6); "
            f"white-space:nowrap; transform:translate(-50%,-50%);'>"
            f"{label}</div>"
        )
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(html=icon_html, icon_size=(0, 0), icon_anchor=(0, 0)),
            interactive=False,
        ).add_to(bubble_layer)

    bubble_layer.add_to(fmap)
    cmap.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)

    # Legend for the bubbles
    fmap.get_root().html.add_child(folium.Element(f"""
    <div style="position: fixed; bottom: 20px; left: 20px; background: white; padding: 10px;
                border: 1px solid #888; border-radius: 4px; font-size: 12px; z-index: 9999;
                max-width: 240px;">
      <b>Bubble legend</b><br>
      Size: |net change %| (max {PCT_MAX:.1f}%).<br>
      <span style="color:#1a9850;">●</span> green: positive net change<br>
      <span style="color:#d73027;">●</span> red: negative net change<br>
      <span style="color:#888888;">●</span> grey: no change / no data
    </div>
    """))

    fmap.save(str(OUT_DIR / output_name))
    print(f"  wrote {output_name}")


# ----------------------------------------------------------------------------
# 5. Plotly map builder
# ----------------------------------------------------------------------------
def build_plotly_map(value_col: str, value_label: str, output_name: str):
    m_data = metrics.dropna(subset=[value_col]).copy()
    vals = m_data[value_col]
    if (vals.min() < 0) and (vals.max() > 0):
        bound = max(abs(vals.quantile(0.05)), abs(vals.quantile(0.95)))
        rng = (-bound, bound)
        scale = "RdBu"
    else:
        rng = (vals.quantile(0.05), vals.quantile(0.95))
        scale = "Reds" if vals.min() >= 0 else "Blues_r"

    fig = px.choropleth_map(
        m_data,
        geojson=geojson,
        locations="KommuneCode",
        featureidkey="properties.kode",
        color=value_col,
        color_continuous_scale=scale,
        range_color=rng,
        map_style="carto-positron",
        zoom=5.8, center={"lat": 56.0, "lon": 10.5},
        opacity=0.70,
        hover_name="KommuneName",
        hover_data={
            "KommuneCode": False,
            value_col: ":+.2f",
            "net_total": ":+,",
            "net_pct": ":+.2f",
        },
        labels={value_col: value_label, "net_total": "Net total (units)",
                "net_pct": "Net change (%)"},
    )

    # Build bubbles. Split into green/red/grey so legend is informative.
    bubble_rows = []
    for _, row in metrics.iterrows():
        code = row["KommuneCode"]
        if code not in centroids:
            continue
        lat, lon = centroids[code]
        np_val = row["net_pct"]
        bubble_rows.append({
            "lat": lat, "lon": lon,
            "net_pct": np_val,
            "radius": bubble_radius(np_val),
            "color": bubble_color(np_val),
            "label": bubble_label(np_val),
            "sign": ("positive" if pd.notna(np_val) and np_val > 0
                     else "negative" if pd.notna(np_val) and np_val < 0
                     else "zero / no data"),
        })
    bdf = pd.DataFrame(bubble_rows)

    for sign, color, label in [("positive", "#1a9850", "Net change ↑"),
                               ("negative", "#d73027", "Net change ↓"),
                               ("zero / no data", "#888888", "No change / no data")]:
        sub = bdf[bdf["sign"] == sign]
        if len(sub) == 0:
            continue
        # Plotly Scattermap accepts an array of sizes per marker
        fig.add_trace(go.Scattermap(
            lat=sub["lat"], lon=sub["lon"],
            mode="markers+text",
            marker=dict(size=sub["radius"] * 2,  # plotly size is diameter-ish
                        color=color, opacity=0.6),
            text=sub["label"],
            textfont=dict(size=11, color="white"),
            textposition="middle center",
            name=label,
            hoverinfo="skip",  # bubbles have no hover; polygon hover wins
        ))

    fig.update_layout(
        title=f"{value_label}, 2008–2024",
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01,
                    bgcolor="rgba(255,255,255,0.85)"),
        height=750,
    )
    fig.write_html(OUT_DIR / output_name, include_plotlyjs="cdn", full_html=True)
    print(f"  wrote {output_name}")


# ----------------------------------------------------------------------------
# 6. Build all four
# ----------------------------------------------------------------------------
print("\nMap 1: Population change")
build_folium_map("pop_pct", "Population change 2008–2024 (%)", "map_population_folium.html")
build_plotly_map("pop_pct", "Population change 2008–2024 (%)", "map_population_plotly.html")

print("\nMap 2: Average income change")
build_folium_map("income_pct", "Avg income change 2008–2024 (%)", "map_income_folium.html")
build_plotly_map("income_pct", "Avg income change 2008–2024 (%)", "map_income_plotly.html")

print(f"\nAll maps saved to {OUT_DIR.resolve()}")

Loading panel...
  99 kommuner with metrics
Fetching GeoJSON...

Map 1: Population change
  wrote map_population_folium.html
  wrote map_population_plotly.html

Map 2: Average income change
  wrote map_income_folium.html
  wrote map_income_plotly.html

All maps saved to /home/edgar/MSc_HCAI_Py/SocialDataFinalProject/notebooks/outputs
